# GPU code-block test (Colab)

**What this does:** runs the 7 Python code blocks of the course that need an NVIDIA GPU
(cell2location, rapids-singlecell, cuGraph/CAGRA, RMM/CuPy/cuML, PyTorch AMP) and records PASS / FAIL, the exception
and the elapsed time for each block in `gpu_results.json`.

**How to run**
1. Menu **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU** (an L4/A100 also works), then Save.
2. **Runtime -> Run all.** Expected duration: about **15-30 min** (installs 8-15 min, RAPIDS being the slowest; the blocks themselves 5-10 min).
3. If Colab says *"Restart session"* after the install cells, click **Restart**, then **Run all** again (installs are idempotent).
4. At the end, download **`gpu_results.json`** (the last cell tries to start the download; otherwise use the Files pane on the left) and send it back.

**Notes**
- Chapter blocks are executed verbatim from the course sources. Everything marked *HARNESS* (synthetic data, calls into functions that
  the block only defines, shrunk epochs / pool sizes for a 16 GB T4) is test scaffolding, not part of the chapters.
- Each block is isolated (own namespace) and wrapped in try/except; one failure does not stop the others.
- Source of truth for the inventory: `review/fix_log_eng.md`, section "GPU blocks (Colab) (2026-10-21)".

## 0. GPU check

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi: not found")
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print("torch", torch.__version__, "| torch CUDA", torch.version.cuda, "| cuda available:", HAS_GPU)
except Exception as e:  # torch is preinstalled on Colab
    HAS_GPU = False
    print("torch import failed:", e)
if not HAS_GPU:
    raise RuntimeError("NO GPU DETECTED. Use Runtime -> Change runtime type -> T4 GPU, then Run all again. Stopping here.")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB)")

## 1. Installs
Grouped by stack. Package names / extras were checked against PyPI metadata on 2026-10-21:
`rapids-singlecell` (extras `rapids-cu12` / `rapids-cu13`; Python >= 3.12), `cell2location` 0.1.5 (requires `scvi-tools>=1.3.0`),
`scvi-tools` 1.5.1. **Run 2 (2026-10-27): `pyro-ppl<1.9.2` is pinned** because run 1 failed at `import cell2location` (pyro-ppl 1.9.2 no longer defines `deep_getattr`). Other versions are left unpinned on purpose so the blocks are tested against what a learner gets today;
the exact versions are recorded in `gpu_results.json`.

### 1a. torch / scvi-tools stack (cell2location, scanpy, transformers)

In [ ]:
!pip install -q scanpy scvi-tools cell2location "pyro-ppl<1.9.2" transformers psutil 2>&1 | tail -3

### 1b. RAPIDS stack (cuDF, cuML, cuGraph, cuVS, RMM, CuPy) via rapids-singlecell
Official route: the `rapids-singlecell` extra for your CUDA major version (pulls `cudf`, `cuml`, `cugraph`, `cuvs`, `cupy`, `rmm` from PyPI).
**Expected time: 5-12 minutes** (large wheels). Colab's CUDA major is detected automatically.

In [ ]:
import re, subprocess
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*(\d+)\.", smi)
cuda_major = int(m.group(1)) if m else 12
extra = "rapids-cu13" if cuda_major >= 13 else "rapids-cu12"
print("driver CUDA major:", cuda_major, "-> extra:", extra)
!pip install -q "rapids-singlecell[{extra}]" --extra-index-url=https://pypi.nvidia.com 2>&1 | tail -5

### 1c. Import check (stack versions are written into the results file)

In [ ]:
import importlib
VERSIONS = {}
for name in ["torch", "scvi", "cell2location", "scanpy", "anndata", "transformers", "cupy", "cudf", "cuml", "cugraph", "cuvs", "rmm", "rapids_singlecell"]:
    try:
        mod = importlib.import_module(name)
        VERSIONS[name] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        VERSIONS[name] = f"IMPORT FAILED: {type(e).__name__}: {str(e)[:150]}"
for k, v in VERSIONS.items():
    print(f"{k:18s} {v}")

## 2. Harness (runs each block in isolation and records the result)

In [ ]:
import time, traceback, json, warnings, platform
RESULTS = {"meta": {}, "blocks": {}}

def run_block(bid, location, src, setup=None, post=None, replace=(), notes=""):
    """Run chapter block `src` verbatim (optionally with documented textual replacements) in a fresh namespace.
    setup(): HARNESS synthetic data;  post(ns): HARNESS calls into functions the block only defines."""
    rec = {"id": bid, "location": location, "status": None, "phase": None, "error": None,
           "traceback": None, "elapsed_s": None, "notes": notes, "replacements": [list(r) for r in replace],
           "warnings": []}
    ns = {"__name__": "__main__"}
    phase = "setup"
    t0 = time.time()
    with warnings.catch_warnings(record=True) as wlist:
        warnings.simplefilter("always")
        try:
            if setup:
                setup()
            phase = "block"
            s = src
            for a, b in replace:
                assert a in s, f"replacement target not found: {a!r}"
                s = s.replace(a, b)
            exec(compile(s, f"<{bid} {location}>", "exec"), ns)
            if post:
                phase = "post"
                post(ns)
            rec["status"] = "PASS"
        except (Exception, SystemExit) as e:
            rec["status"] = "SETUP_FAIL" if phase == "setup" else "FAIL"
            rec["error"] = f"{type(e).__name__}: {str(e)[:600]}"
            rec["traceback"] = traceback.format_exc()[-2500:]
        finally:
            rec["phase"] = phase
            rec["elapsed_s"] = round(time.time() - t0, 2)
            seen = set()
            for w in wlist:
                msg = f"{w.category.__name__}: {str(w.message)[:200]}"
                if msg not in seen and len(seen) < 12:
                    seen.add(msg)
            rec["warnings"] = sorted(seen)
    RESULTS["blocks"][bid] = rec
    print(f"[{rec['status']}] {bid} ({location}) phase={rec['phase']} {rec['elapsed_s']}s" + (f"\n    {rec['error']}" if rec["error"] else ""))
    return rec

## 3. Blocks

### B1  `tracks/05_generative_foundation_models/docs/00_FOUNDATION_MODELS_ROADMAP.md:182` (Track 05 ch00 — environment diagnostic; runnable alone)

Runnable alone (needs torch, scvi, scanpy, transformers). The chapter's own `[PASS]/[FAIL]` printout is shown, and the cell passes if the script runs to the end.

In [ ]:
B1_SRC = r'''"""
00_verify_environment.py
단일세포 생성형 파운데이션 모델 전용 하드웨어 가속 및 의존성 검증 스크립트.
"""

import sys
import platform
import importlib

def check_single_cell_dl_stack():
    print(f"=== Single-Cell Deep Learning Environment Diagnostic ===")
    print(f"Python Version : {sys.version.split()[0]}")
    print(f"Platform       : {platform.platform()}")
    print("-" * 55)

    # 1. 핵심 패키지 임포트 검증
    required_packages = [
        ("torch", "PyTorch Core Engine"),
        ("scvi", "scvi-tools (Generative Models)"),
        ("scanpy", "Scanpy (Single-Cell Toolkit)"),
        ("anndata", "AnnData Data Structure"),
        ("transformers", "Hugging Face Transformers (Geneformer용; scGPT만 쓰면 선택)"),
        ("scipy", "SciPy Scientific Computing"),
        ("numpy", "NumPy Matrix Operations")
    ]

    all_passed = True
    for pkg_name, description in required_packages:
        try:
            mod = importlib.import_module(pkg_name)
            ver = getattr(mod, "__version__", "unknown")
            print(f" [PASS] {pkg_name:<15} v{ver:<10} - {description}")
        except ImportError as e:
            print(f" [FAIL] {pkg_name:<15} NOT FOUND   - {description} ({e})")
            all_passed = False

    print("-" * 55)

    # 2. PyTorch 하드웨어 텐서 가속 검증
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    
    device_found = False
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f" [ACCELERATOR] NVIDIA CUDA Active:")
        print(f"   - GPU Device : {gpu_name} (Total: {gpu_count} GPU(s))")
        print(f"   - Total VRAM : {vram_gb:.2f} GB")
        print(f"   - CUDA Arch  : {torch.version.cuda}")
        device_found = True
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print(f" [ACCELERATOR] Apple Silicon MPS (Metal Performance Shaders) Active")
        device_found = True
        device = torch.device("mps")
    else:
        print(f" [WARNING] No Hardware Acceleration Detected. Fallback to CPU.")
        device = torch.device("cpu")

    # 3. 텐서 연산 벤치마크 테스트 (Negative Binomial Log-Likelihood)
    x = torch.randint(low=0, high=100, size=(1000, 2000), device=device).float()
    mu = torch.rand(1000, 2000, device=device) * 50.0 + 1e-4
    theta = torch.ones(2000, device=device) * 2.0
    
    # NB log prob 수치 테스트
    from torch.distributions import NegativeBinomial
    # torch NegativeBinomial의 평균은 total_count * p / (1 - p) 이므로 평균을 mu로 만들려면 p = mu / (theta + mu)
    probs = mu / (theta + mu)
    nb_dist = NegativeBinomial(total_count=theta, probs=probs)
    log_probs = nb_dist.log_prob(x)
    
    print(f" [TEST] Negative Binomial Likelihood Tensor Computation:")
    print(f"   - Input Shape : {tuple(x.shape)}")
    print(f"   - Device Used : {x.device}")
    print(f"   - Mean Log-Likelihood: {log_probs.mean().item():.4f}")
    print("-" * 55)
    
    if all_passed and device_found:
        print(">>> 시스템 준비 완료: 모든 생성형 파운데이션 모델 학습 가능.")
    else:
        print(">>> 환경 구성 확인 필요: 미흡한 의존성 또는 CPU 전용 모드.")

if __name__ == "__main__":
    check_single_cell_dl_stack()
'''

run_block("B1", "tracks/05_generative_foundation_models/docs/00_FOUNDATION_MODELS_ROADMAP.md:182", B1_SRC)

### B2  `tracks/05_generative_foundation_models/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:139` (Track 05 ch09 — AMP training loop + simplified scIB; runnable alone, plus GPU smoke test)

The chapter block runs alone (CPU metrics in its `__main__`). The block only *defines* `train_single_cell_transformer_amp`, so the HARNESS additionally calls it on a toy model on the GPU (this exercises `torch.cuda.amp.GradScaler/autocast`, which newer PyTorch deprecates).

In [ ]:
B2_SRC = r'''import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from typing import Dict, Tuple

def train_single_cell_transformer_amp(
    model: nn.Module,
    dataloader,
    optimizer: torch.optim.Optimizer,
    num_epochs: int = 5,
    gradient_accumulation_steps: int = 4,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
) -> nn.Module:
    """
    Automatic Mixed Precision (AMP) 및 Gradient Accumulation이 적용된 트랜스포머 훈련 루프.
    """
    model.to(device)
    model.train()
    
    # FP16/BF16 그래디언트 스케일러 초기화
    # 주의: GradScaler는 FP16 용이다. BF16 autocast를 쓸 때는 필요 없다.
    scaler = GradScaler(enabled=(device == "cuda"))
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        running_loss = 0.0
        optimizer.zero_grad()

        for step, (tokens, values, labels) in enumerate(dataloader):
            tokens = tokens.to(device)
            values = values.to(device)
            labels = labels.to(device)

            # 1. AMP Context Manager를 통한 반정밀도 순전파
            with autocast(enabled=(device == "cuda")):
                _, logits = model(tokens, values)
                loss = criterion(logits, labels)
                # 그래디언트 누적 스텝으로 나눔
                loss = loss / gradient_accumulation_steps

            # 2. Scaled Backward Pass
            scaler.scale(loss).backward()
            running_loss += loss.item() * gradient_accumulation_steps

            # 3. 누적 스텝 도달 시 파라미터 업데이트
            if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        avg_loss = running_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] - Mean Loss: {avg_loss:.4f}")

    return model


def compute_simplified_scib_metrics(
    embeddings: np.ndarray,
    cell_types: np.ndarray,
    batches: np.ndarray,
    k_neighbors: int = 30
) -> Dict[str, float]:
    """
    추출된 세포 임베딩에 대해 NMI, ASW, 간이 iLISI, cLISI 지표를 계산합니다.
    교육용 간이 점수이며 scIB 공식 정규화와 다르다. 난수 임베딩도 중간 정도의 종합 점수(아래 __main__ 난수 입력에서 약 0.6)를 받으므로 실제 평가에서는 scib-metrics를 쓰고 난수/PCA 베이스라인과 비교한다.
    """
    n_samples = embeddings.shape[0]
    
    # 1. ASW (Silhouette Score) 계산
    asw_celltype = (silhouette_score(embeddings, cell_types) + 1.0) / 2.0
    asw_batch = 1.0 - abs(silhouette_score(embeddings, batches))

    # 2. k-NN 기반 LISI 근사 계산
    nn_model = NearestNeighbors(n_neighbors=k_neighbors).fit(embeddings)
    _, indices = nn_model.kneighbors(embeddings)

    ilisi_scores = []
    clisi_scores = []

    for i in range(n_samples):
        nbrs = indices[i]
        
        # Batch Simpson's Index
        batch_counts = pd.Series(batches[nbrs]).value_counts(normalize=True).values
        simpson_batch = (batch_counts ** 2).sum()
        ilisi_scores.append(1.0 / (simpson_batch + 1e-8))

        # Cell-type Simpson's Index
        celltype_counts = pd.Series(cell_types[nbrs]).value_counts(normalize=True).values
        simpson_celltype = (celltype_counts ** 2).sum()
        clisi_scores.append(1.0 / (simpson_celltype + 1e-8))

    mean_ilisi = float(np.mean(ilisi_scores))
    mean_clisi = float(np.mean(clisi_scores))

    # 2.5 NMI: 임베딩을 KMeans(k=세포형 수)로 군집화한 결과와 세포형 라벨 간 NMI
    km = KMeans(n_clusters=len(np.unique(cell_types)), n_init=10, random_state=0).fit_predict(embeddings)
    nmi = normalized_mutual_info_score(cell_types, km)

    # 3. 종합 점수 가중 합산
    bio_score = (asw_celltype + nmi + (1.0 / mean_clisi)) / 3.0 # clisi는 1에 가까울수록 완벽
    batch_score = 0.5 * asw_batch + 0.5 * (mean_ilisi / len(np.unique(batches)))
    total_score = 0.6 * bio_score + 0.4 * batch_score

    return {
        "ASW_CellType (Bio)": asw_celltype,
        "ASW_Batch (Batch)": asw_batch,
        "mean_iLISI (Batch)": mean_ilisi,
        "mean_cLISI (Bio)": mean_clisi,
        "NMI_CellType (Bio)": nmi,
        "Overall_Score": total_score
    }


# 테스트 및 검증 실행
if __name__ == "__main__":
    np.random.seed(42)
    N = 200
    D = 64
    
    # 모의 임베딩, 세포형(2종: Adeno vs NEPC), 배치(2개 병원)
    mock_emb = np.random.randn(N, D)
    mock_celltypes = np.array(["AR-PC"] * 100 + ["t-NEPC"] * 100)
    mock_batches = np.array(["Hospital_A", "Hospital_B"] * 100)

    metrics = compute_simplified_scib_metrics(mock_emb, mock_celltypes, mock_batches)
    print("\n[단일 세포 파운데이션 모델 scIB 벤치마킹 산출 결과]")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
'''

def _post_B2(ns):
    import torch, torch.nn as nn
    class Toy(nn.Module):  # HARNESS toy model: forward(tokens, values) -> (embedding, logits)
        def __init__(self):
            super().__init__(); self.e = nn.Embedding(50, 16); self.l = nn.Linear(16, 3)
        def forward(self, t, v):
            h = (self.e(t) * v.unsqueeze(-1)).mean(1); return h, self.l(h)
    data = [(torch.randint(0, 50, (8, 10)), torch.rand(8, 10), torch.randint(0, 3, (8,))) for _ in range(8)]
    m = Toy(); opt = torch.optim.Adam(m.parameters(), 1e-3)
    out = ns["train_single_cell_transformer_amp"](m, data, opt, num_epochs=2, gradient_accumulation_steps=4, device="cuda")
    assert next(out.parameters()).is_cuda

run_block("B2", "tracks/05_generative_foundation_models/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:139", B2_SRC, post=_post_B2)

### B3  `tracks/02_spatial_transcriptomics/docs/05_SPATIAL_DECONVOLUTION_MATHEMATICS.md:222` (Track 02 ch05 — cell2location (use_gpu=True); needs synthetic setup)

The block reads `data/mcrpc_reference_snrnaseq.h5ad` and `data/visium_prostate_cancer.h5ad`; the HARNESS writes small synthetic versions (600 reference cells, 150 Visium spots, 200 genes, fake H&E image). **Documented replacement:** `max_epochs` 250 -> 20 and 30000 -> 300 so the cell finishes in minutes (set `FULL_EPOCHS = True` to run the chapter values; that can take 15+ min). Run 1 showed `use_gpu=True` is rejected by the installed scvi-tools Trainer; the chapter now uses `accelerator="gpu"` (also inside `sample_kwargs`).

In [ ]:
B3_SRC = r'''import scanpy as sc
import anndata as ad
import cell2location
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# Step 1: snRNA-seq 기준 데이터로부터 세포 유형별 시그니처 추출
# ---------------------------------------------------------
adata_ref = sc.read_h5ad("data/mcrpc_reference_snrnaseq.h5ad")

# 1.1 시그니처 추정에 사용할 비중복 정보 유전자(HVG) 선별
cell2location.models.RegressionModel.setup_anndata(
    adata=adata_ref,
    batch_key="patient_id",       # 환자별 배치 효과 보정
    labels_key="lineage_4states", # 4-state 계통 (AR+, Amphicrine, DNPC, t-NEPC, CAFs 등)
    categorical_covariate_keys=["technology"]
)

# 1.2 음이항 회귀 모델 학습 (Stage 1)
reg_model = cell2location.models.RegressionModel(adata_ref)
reg_model.train(max_epochs=250, accelerator="gpu")

# 1.3 세포 유형별 단위 세포당 발현율 (g_{k, g}) 추출
adata_ref = reg_model.export_posterior(
    adata_ref, sample_kwargs={"num_samples": 1000, "batch_size": 2500, "accelerator": "gpu"}
)
inf_aver = adata_ref.varm["means_per_cluster_mu_fg"][[
    f"means_per_cluster_mu_fg_{k}" for k in adata_ref.obs["lineage_4states"].cat.categories
]]
inf_aver.columns = adata_ref.obs["lineage_4states"].cat.categories

# ---------------------------------------------------------
# Step 2: Visium 공간전사체 스팟 디콘볼루션 (Stage 2)
# ---------------------------------------------------------
adata_vis = sc.read_h5ad("data/visium_prostate_cancer.h5ad")

# 2.1 공통 유전자 교집합 정렬
intersect_genes = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, intersect_genes].copy()
inf_aver = inf_aver.loc[intersect_genes, :]

# 2.2 조직학적 사전 정보(H&E 평균 핵 개수) 설정
# 전립선암 H&E 슬라이드 검사 결과 스팟당 평균 약 15개의 세포가 관측됨
cell2location.models.Cell2location.setup_anndata(
    adata=adata_vis,
    batch_key="sample_slide_id"
)

# 2.3 cell2location 모델 인스턴스 생성 및 사전분포 주입
c2l_model = cell2location.models.Cell2location(
    adata_vis,
    cell_state_df=inf_aver,
    N_cells_per_location=15,      # 스팟(location)당 평균 세포 수 하이퍼파라미터 (인자명이 N_cells_per_location; 기본값 8)
    detection_alpha=20            # 스팟별 검출 민감도 y_s의 감마 분포 정규화 강도
)

# 2.4 변분 추론(VI) 학습 실행
c2l_model.train(
    max_epochs=30000,
    batch_size=None,              # 전체 스팟 일괄 학습
    train_size=1.0,
    accelerator="gpu"
)

# ---------------------------------------------------------
# Step 3: 사후분포 추출 및 보수적 절대 세포 수(q05) 도출
# ---------------------------------------------------------
adata_vis = c2l_model.export_posterior(
    adata_vis, sample_kwargs={"num_samples": 1000, "batch_size": 2048, "accelerator": "gpu"}
)

# 사후분포 5% 분위수(q05)를 관측치 메타데이터(obs)에 추가
# obsm 키는 'q05_cell_abundance_w_sf', 열 이름은 'q05cell_abundance_w_sf_<세포유형>' ('q05' 뒤에 밑줄 없음)
adata_vis.obs[adata_vis.obsm["q05_cell_abundance_w_sf"].columns] = (
    adata_vis.obsm["q05_cell_abundance_w_sf"].values
)

# ---------------------------------------------------------
# Step 4: 공간적 세포 밀도 분포 시각화
# ---------------------------------------------------------
with plt.rc_context({"figure.figsize": (6, 6)}):
    sc.pl.spatial(
        adata_vis,
        color=[
            "q05cell_abundance_w_sf_Luminal_AR_plus",
            "q05cell_abundance_w_sf_t_NEPC",
            "q05cell_abundance_w_sf_myCAF",
            "q05cell_abundance_w_sf_CD8_T_cell"
        ],
        cmap="magma",
        ncols=2,
        size=1.3,
        img_key="hires"
    )
'''

FULL_EPOCHS = False
def _setup_B3():
    import numpy as np, pandas as pd, anndata as ad, os, scipy.sparse as sp
    rng = np.random.default_rng(0); os.makedirs("data", exist_ok=True)
    G = 200; genes = [f"GENE{i}" for i in range(G)]
    types = ["Luminal_AR_plus", "t_NEPC", "myCAF", "CD8_T_cell", "Basal"]
    sig = rng.gamma(1.0, 1.0, (len(types), G)) + 0.2
    ct = rng.integers(0, len(types), 600)
    X = rng.poisson(sig[ct] * 3).astype("float32")
    ref = ad.AnnData(sp.csr_matrix(X), obs=pd.DataFrame({
        "patient_id": rng.choice(["P1", "P2", "P3"], 600),
        "lineage_4states": pd.Categorical(np.array(types)[ct], categories=types),
        "technology": rng.choice(["A", "B"], 600)}, index=[f"r{i}" for i in range(600)]),
        var=pd.DataFrame(index=genes))
    ref.write_h5ad("data/mcrpc_reference_snrnaseq.h5ad")
    S = 150; mix = rng.dirichlet(np.ones(len(types)), S)
    V = rng.poisson(mix @ sig * 15).astype("float32")
    vis = ad.AnnData(sp.csr_matrix(V), obs=pd.DataFrame({"sample_slide_id": rng.choice(["S1", "S2"], S)}, index=[f"s{i}" for i in range(S)]),
                     var=pd.DataFrame(index=genes))
    vis.obsm["spatial"] = rng.uniform(10, 90, (S, 2))
    vis.uns["spatial"] = {"lib": {"images": {"hires": rng.uniform(0.6, 1, (100, 100, 3)).astype("float32")},
                                  "scalefactors": {"tissue_hires_scalef": 1.0, "spot_diameter_fullres": 4.0}}}
    vis.write_h5ad("data/visium_prostate_cancer.h5ad")

run_block("B3", "tracks/02_spatial_transcriptomics/docs/05_SPATIAL_DECONVOLUTION_MATHEMATICS.md:222", B3_SRC, setup=_setup_B3,
          replace=[] if FULL_EPOCHS else [("max_epochs=250", "max_epochs=20"), ("max_epochs=30000", "max_epochs=300")])

### B4  `tracks/08_large_scale_sparse/docs/07_GPU_ACCELERATION_RAPIDS_SINGLECELL.md:184` (Track 08 ch07 — rapids-singlecell pipeline (functions only); needs synthetic setup + call)

The block only defines `init_gpu_memory_pool`, `load_1m_atlas_to_gpu`, `run_gpu_accelerated_pipeline`. The HARNESS builds a small synthetic atlas (6000 cells x 4000 genes, with `MT-` genes), writes it to h5ad, and calls the three functions in order. **Documented replacement of arguments:** RMM pool `60GB/76GB` -> `1GB/4GB` (the chapter defaults target an 80 GB GPU; a T4 has 16 GB). If this block fails with the *default* pool sizes only, that is expected on a small GPU.

In [ ]:
B4_SRC = r'''"""
End-to-End GPU Single-Cell Acceleration Pipeline with rapids-singlecell
Benchmarked on 1,000,000 Prostate Cancer Single-Cell Atlas
"""

import time
import os
import gc
import numpy as np
import scipy.sparse as sp
import cupy as cp
import cupyx.scipy.sparse as cpsp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import rapids_singlecell as rsc
import scanpy as sc
import anndata as ad


def init_gpu_memory_pool(initial_pool_size: str = "60GB", max_pool_size: str = "76GB"):
    """
    RAPIDS 메모리 매니저(RMM) 풀을 초기화하여 CudaMalloc 오버헤드와 단편화를 원천 차단한다.
    """
    print(f"[RMM Init] Allocating GPU Memory Pool ({initial_pool_size} ~ {max_pool_size})...")
    rmm.reinitialize(
        pool_allocator=True,
        initial_pool_size=initial_pool_size,  # RMM이 "60GB" 형식 문자열을 직접 해석 (GB=10^9, GiB=2^30 bytes)
        maximum_pool_size=max_pool_size
    )
    # CuPy의 기본 메모리 할당자를 RMM으로 바인딩
    cp.cuda.set_allocator(rmm_cupy_allocator)
    print("[RMM Init] GPU Memory Pool successfully bound to CuPy & RAPIDS.")


def load_1m_atlas_to_gpu(h5ad_path: str) -> ad.AnnData:
    """
    1M 전립선암 H5AD 파일을 디스크에서 로드하고 AnnData의 카운트 행렬을 GPU CuPy CSR로 즉각 전송한다.
    """
    t0 = time.time()
    print(f"[I/O] Reading H5AD from {h5ad_path}...")
    adata = sc.read_h5ad(h5ad_path)
    print(f"[I/O] Host read completed in {time.time() - t0:.2f}s. Shape: {adata.shape}")

    t_gpu = time.time()
    print("[PCIe] Promoting sparse count matrix to GPU VRAM (cupyx.scipy.sparse.csr_matrix)...")
    if sp.issparse(adata.X):
        # SciPy CSR을 CuPy CSR로 비동기 복사
        adata.X = cpsp.csr_matrix(adata.X, dtype=cp.float32)
    else:
        adata.X = cp.asarray(adata.X, dtype=cp.float32)

    print(f"[PCIe] Transfer completed in {time.time() - t_gpu:.2f}s.")
    return adata


def run_gpu_accelerated_pipeline(adata: ad.AnnData) -> ad.AnnData:
    """
    1M 전립선암 세포에 대해 전처리부터 UMAP까지 GPU 메모리 내에서 연속 체이닝 실행
    """
    total_t0 = time.time()
    print("\n" + "="*65)
    print("   STARTING RAPIDS-SINGLECELL 1M CELLS GPU PIPELINE")
    print("="*65)

    # 1. GPU 기반 고속 품질 관리 (QC Metrics)
    t0 = time.time()
    # 미토콘드리아 유전자 플래그 등록
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    rsc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], log1p=False)
    print(f"[Stage 1] GPU QC Metrics calculation : {time.time() - t0:.3f}s")

    # 2. 세포 및 유전자 필터링 (In-place Boolean Masking on GPU)
    t0 = time.time()
    rsc.pp.filter_cells(adata, min_genes=200)
    rsc.pp.filter_genes(adata, min_cells=10)
    print(f"[Stage 2] GPU Filter Cells & Genes   : {time.time() - t0:.3f}s (Remaining: {adata.n_obs:,} cells)")

    # 3. 초고속 고변동 유전자(HVG) 선별 (Warp Shuffle Reduction)
    #    flavor="seurat_v3"는 정규화·로그 변환 전의 원시 카운트를 요구한다 (scanpy/rsc 문서)
    t0 = time.time()
    rsc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3")
    print(f"[Stage 3] GPU Highly Variable Genes  : {time.time() - t0:.3f}s")

    # 4. 라이브러리 크기 정규화 및 Log1p 변환 (GPU Parallel Kernel)
    t0 = time.time()
    rsc.pp.normalize_total(adata, target_sum=1e4)
    rsc.pp.log1p(adata)
    print(f"[Stage 4] GPU Normalize & Log1p      : {time.time() - t0:.3f}s")

    # 5. HVG 부분집합 선택 후 유전자별 표준화(Scaling) 및 클리핑
    #    부분집합 없이 전체 35,000개 유전자에 scale을 적용하면 1M x 35k 밀집 행렬(약 140 GB)이 되어 VRAM을 초과한다
    t0 = time.time()
    adata = adata[:, adata.var["highly_variable"]].copy()
    rsc.pp.scale(adata, max_value=10.0)
    print(f"[Stage 5] GPU Matrix Scaling (HVG)   : {time.time() - t0:.3f}s")

    # 6. GPU 주성분 분석 (rsc.pp.pca; 구버전은 rsc.tl.pca)
    t0 = time.time()
    rsc.pp.pca(adata, n_comps=50, zero_center=True)
    print(f"[Stage 6] GPU PCA (50 components)    : {time.time() - t0:.3f}s")

    # 7. GPU 근사 최근접 이웃 탐색 (kNN Graph Construction via cuML)
    t0 = time.time()
    rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, metric="euclidean")
    print(f"[Stage 7] GPU kNN Graph (k=15, 1M)   : {time.time() - t0:.3f}s")

    # 8. cuML 병렬 비동기 SGD UMAP 2D 임베딩
    t0 = time.time()
    rsc.tl.umap(adata, min_dist=0.3, spread=1.0)
    print(f"[Stage 8] cuML UMAP 2D Embedding     : {time.time() - t0:.3f}s")

    total_time = time.time() - total_t0
    print("="*65)
    print(f"[SUCCESS] Complete 1M Atlas GPU Pipeline Finished in: {total_time:.2f} seconds!")
    print("="*65 + "\n")

    return adata
'''

def _post_B4(ns):
    import numpy as np, scipy.sparse as sp, anndata as ad, pandas as pd
    rng = np.random.default_rng(1); n, g = 6000, 4000
    X = sp.csr_matrix(rng.poisson(rng.gamma(0.5, 1.0, g) * 1.5, (n, g)).astype("float32"))
    names = [f"MT-{i}" if i < 13 else f"G{i}" for i in range(g)]
    a = ad.AnnData(X, obs=pd.DataFrame(index=[f"c{i}" for i in range(n)]), var=pd.DataFrame(index=names))
    a.write_h5ad("synthetic_atlas.h5ad")
    ns["init_gpu_memory_pool"](initial_pool_size="1GB", max_pool_size="4GB")   # HARNESS: smaller pool
    a = ns["load_1m_atlas_to_gpu"]("synthetic_atlas.h5ad")
    a = ns["run_gpu_accelerated_pipeline"](a)
    assert "X_umap" in a.obsm

run_block("B4", "tracks/08_large_scale_sparse/docs/07_GPU_ACCELERATION_RAPIDS_SINGLECELL.md:184", B4_SRC, post=_post_B4)

### B5  `tracks/08_large_scale_sparse/docs/08_GPU_GRAPH_COMMUNITY_DETECTION.md:171` (Track 08 ch08 — CAGRA kNN + cuGraph Leiden (functions only); needs synthetic setup + call)

The block itself says the CAGRA part was *not run on a GPU* (`GPU 미실행`), so this is the first real check of `cuvs.neighbors.cagra` argument names and return types. The HARNESS calls the three defined functions on 20,000 random 50-d points (float32).

In [ ]:
B5_SRC = r'''"""
Large-Scale Single-Cell Graph Construction & Community Detection
NVIDIA CAGRA + cuGraph Leiden on 1,000,000 Prostate Cancer Atlas
"""

import time
import numpy as np
import cupy as cp
import cudf
import cugraph
# CAGRA는 RAFT(pylibraft)에서 RAPIDS cuVS로 이전되었다 (cuvs.neighbors.cagra; cuVS 소스의 시그니처 기준. 합성 100k점으로 A100에서 실행 확인, 100만 규모는 미실행)
from cuvs.common import Resources
from cuvs.neighbors import cagra
import scipy.sparse as sp
import cupyx.scipy.sparse as cpsp
from typing import Tuple


def build_cagra_knn_graph_1m(X_pca_cp: cp.ndarray, k: int = 15) -> Tuple[cp.ndarray, cp.ndarray]:
    """
    NVIDIA CAGRA를 활용하여 100만 세포(50 PCs)의 kNN 그래프를 GPU 네이티브로 초고속 구축한다.
    
    Parameters:
        X_pca_cp: GPU VRAM에 상주하는 CuPy float32 배열 (1,000,000 x 50)
        k: 탐색할 최근접 이웃 수 (기본 15)
        
    Returns:
        knn_indices: 각 세포별 k개 이웃의 정점 인덱스 (1,000,000 x k, int32)
        knn_distances: 유클리드 거리 행렬 (1,000,000 x k, float32)
    """
    N, d = X_pca_cp.shape
    print(f"\n[CAGRA] Initializing Fixed Out-Degree Graph Index for {N:,} cells (dim={d})...")
    t0 = time.time()
    
    handle = Resources()
    
    # 1. CAGRA 인덱스 빌드 파라미터 구성
    build_params = cagra.IndexParams(
        intermediate_graph_degree=64,   # 중간 탐색 그래프 차수
        graph_degree=32,                # 최종 고정 차수(Fixed Out-degree)
        build_algo="ivf_pq"             # 초기 kNN 그래프 구축 알고리즘(ivf_pq 또는 nn_descent)
    )
    
    # 2. 인덱스 구축 (GPU 상에서 연속 실행)
    index = cagra.build(build_params, X_pca_cp, resources=handle)
    build_time = time.time() - t0
    print(f"[CAGRA] Index Construction completed in {build_time:.2f}s.")
    
    # 3. kNN 질의(Query) 탐색 파라미터 구성
    t_search = time.time()
    search_params = cagra.SearchParams(
        max_queries=0,                  # 질의 배치 크기 (0 = 자동 선택; 100_000으로 고정하면 질의 100k에서 CUDA 오류, A100 실측)
        itopk_size=64                   # 내부 우선순위 큐 크기
    )
    
    # 4. 전체 100만 세포에 대한 배치 질의 실행
    # 질의=데이터셋이므로 각 세포의 이웃 목록에는 자기 자신이 포함된다 → k+1개를 질의한 뒤 자기 자신을 제거한다
    # (반환값은 CuPy가 아니라 device_ndarray이므로 cp.asarray로 변환; 기본 metric은 sqeuclidean이므로 거리는 제곱 거리)
    distances, neighbors = cagra.search(
        search_params, index, X_pca_cp, k + 1, resources=handle
    )
    nbr = cp.asarray(neighbors)
    dist = cp.sqrt(cp.maximum(cp.asarray(distances), 0.0))
    is_self = nbr == cp.arange(N, dtype=nbr.dtype)[:, None]
    is_self[:, -1] |= ~is_self.any(axis=1)      # 근사 탐색에서 자기 자신이 누락된 행은 가장 먼 이웃 1개를 제거
    keep = ~is_self
    knn_indices = nbr[keep].reshape(N, k)
    knn_distances = dist[keep].reshape(N, k)
    search_time = time.time() - t_search
    print(f"[CAGRA] kNN Search (k={k}) for 1M cells completed in {search_time:.2f}s.")
    print(f"[CAGRA] Total Graph Time: {time.time() - t0:.2f}s.")
    
    return knn_indices, knn_distances


def convert_knn_to_cugraph(knn_indices: cp.ndarray, knn_distances: cp.ndarray) -> cugraph.Graph:
    """
    kNN 탐색 결과를 cuGraph Graph 객체(CSR / Edge List)로 변환한다.
    가중치는 UMAP 스타일 퍼지 지수 감쇠(Fuzzy Simplicial Set Kernel)를 적용.
    """
    t0 = time.time()
    N, k = knn_indices.shape
    print(f"[Graph Prep] Converting kNN tensors into cuGraph Edge List (Total Edges: {N*k:,})...")
    
    # 1. Source 및 Destination 노드 벡터 구성
    src_nodes = cp.repeat(cp.arange(N, dtype=cp.int32), k)
    dst_nodes = knn_indices.ravel().astype(cp.int32)
    
    # 2. 거리 기반 지수 가중치 계산: w_ij = exp(-d_ij)
    weights = cp.exp(-cp.maximum(knn_distances.ravel() - 1e-3, 0.0)).astype(cp.float32)
    
    # 3. cuDF GPU DataFrame 생성 (Zero-Copy)
    edge_df = cudf.DataFrame({
        "source": src_nodes,
        "destination": dst_nodes,
        "weight": weights
    })
    
    # 4. 무방향 가중치 그래프 인스턴스화
    G = cugraph.Graph(directed=False)
    G.from_cudf_edgelist(
        edge_df, source="source", destination="destination", edge_attr="weight"
    )
    print(f"[Graph Prep] cuGraph initialized in {time.time() - t0:.2f}s. Number of Vertices: {G.number_of_vertices():,}")
    return G


def run_cugraph_leiden_clustering(G: cugraph.Graph, resolution: float = 0.8, 
                                  max_iter: int = 50) -> cudf.DataFrame:
    """
    cuGraph 병렬 Leiden 알고리즘을 실행하여 100만 세포의 커뮤니티 분할을 수행한다.
    """
    print(f"\n[Leiden] Starting cuGraph Parallel Leiden (Resolution: {resolution}, Max Iter: {max_iter})...")
    t0 = time.time()
    
    # GPU 병렬 Leiden 커널 디스패치 (내부 충돌 회피 방식은 cuGraph 구현에 따름)
    parts, modularity_score = cugraph.leiden(
        G,
        resolution=resolution,
        max_iter=max_iter
    )
    
    elapsed = time.time() - t0
    n_clusters = parts["partition"].nunique()
    print("="*65)
    print(f"[Leiden Success] Clustered {G.number_of_vertices():,} cells into {n_clusters} communities!")
    print(f"Final Graph Modularity (Q) : {modularity_score:.5f}")
    print(f"Total Leiden Runtime       : {elapsed:.3f} seconds")
    print("="*65 + "\n")
    
    return parts
'''

def _post_B5(ns):
    import cupy as cp
    cp.random.seed(0)
    X = cp.random.standard_normal((20000, 50), dtype=cp.float32)   # HARNESS synthetic PCs
    idx, dist = ns["build_cagra_knn_graph_1m"](X, k=15)
    assert idx.shape == (20000, 15) and dist.shape == (20000, 15)
    G = ns["convert_knn_to_cugraph"](idx, dist)
    parts = ns["run_cugraph_leiden_clustering"](G, resolution=0.8, max_iter=50)
    assert len(parts) == 20000

run_block("B5", "tracks/08_large_scale_sparse/docs/08_GPU_GRAPH_COMMUNITY_DETECTION.md:171", B5_SRC, post=_post_B5)

### B7  `tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:205` (Track 08 ch09 — memory guard + cuML PCA (functions only); needs synthetic setup + call)

The block defines a decorator and `execute_safe_pca`; the HARNESS calls `execute_safe_pca` on a random 5000 x 100 CuPy matrix.

In [ ]:
B7_SRC = r'''"""
Enterprise Single-Cell Pipeline Diagnostic & Memory Guard Module
Real-time VRAM/RAM Monitoring and Automatic Fragmentation Defense
"""

import functools
import gc
import time
import os
import psutil
import cupy as cp
import rmm


def single_cell_memory_guard(stage_name: str, warn_vram_gb: float = 70.0):
    """
    단일세포 파이프라인의 각 단계 전후로 CPU RAM과 GPU VRAM의 누수 및 단편화를 정밀 추적하고
    비정상적인 메모리 잔류를 강제 회수하는 방어적 데코레이터.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # 1. 실행 전 메모리 상태 스냅샷
            process = psutil.Process(os.getpid())
            ram_before_mb = process.memory_info().rss / (1024 * 1024)
            
            # GPU VRAM 상태 쿼리
            free_bytes, total_bytes = cp.cuda.runtime.memGetInfo()
            vram_used_before_gb = (total_bytes - free_bytes) / (1024**3)
            total_vram_gb = total_bytes / (1024**3)

            print(f"\n[MemoryGuard: START] >>> Stage: {stage_name}")
            print(f"  CPU RAM Baseline : {ram_before_mb:.2f} MB")
            print(f"  GPU VRAM Used    : {vram_used_before_gb:.2f} GB / {total_vram_gb:.2f} GB")

            t0 = time.time()
            try:
                result = func(*args, **kwargs)
            except cp.cuda.memory.OutOfMemoryError as oom:
                print(f"\n[CRITICAL OOM DETECTED] Stage '{stage_name}' crashed with CUDA Out of Memory!")
                print("[Emergency Recovery] Flushing CuPy Memory Pool and forcing Python GC...")
                # 긴급 메모리 회수
                gc.collect()
                cp.get_default_memory_pool().free_all_blocks()
                raise oom

            elapsed = time.time() - t0

            # 2. 실행 후 메모리 상태 스냅샷
            ram_after_mb = process.memory_info().rss / (1024 * 1024)
            free_bytes, _ = cp.cuda.runtime.memGetInfo()
            vram_used_after_gb = (total_bytes - free_bytes) / (1024**3)
            vram_delta_gb = vram_used_after_gb - vram_used_before_gb

            print(f"[MemoryGuard: END] <<< Stage: {stage_name} (Duration: {elapsed:.2f}s)")
            print(f"  CPU RAM Delta    : {ram_after_mb - ram_before_mb:+.2f} MB (Current: {ram_after_mb:.2f} MB)")
            print(f"  GPU VRAM Delta   : {vram_delta_gb:+.2f} GB (Current: {vram_used_after_gb:.2f} GB)")

            # 3. 위험 임계치 경고 및 방어 조치
            if vram_used_after_gb >= warn_vram_gb:
                print(f"  [WARNING] VRAM utilization exceeds {warn_vram_gb} GB! Performing defensive defragmentation...")
                gc.collect()
                cp.get_default_memory_pool().free_all_blocks()
                free_bytes_post, _ = cp.cuda.runtime.memGetInfo()
                reclaimed_gb = ((total_bytes - free_bytes_post) - vram_used_after_gb)
                print(f"  [Recovery Complete] Reclaimed {abs(reclaimed_gb):.2f} GB of fragmented VRAM.")

            return result
        return wrapper
    return decorator


# 사용 예시
@single_cell_memory_guard(stage_name="cuML PCA 50 PCs", warn_vram_gb=65.0)
def execute_safe_pca(X_gpu, n_components=50):
    import cuml
    # cuML PCA의 svd_solver는 'full' | 'jacobi' | 'auto' 만 지원한다 ('randomized'는 없음; 랜덤화 SVD는 rsc.pp.pca(..., svd_solver="randomized") 사용)
    pca = cuml.PCA(n_components=n_components, svd_solver="jacobi")
    return pca.fit_transform(X_gpu)
'''

def _post_B7(ns):
    import cupy as cp
    X = cp.random.standard_normal((5000, 100), dtype=cp.float32)   # HARNESS synthetic matrix
    out = ns["execute_safe_pca"](X, n_components=10)
    assert out.shape == (5000, 10)

run_block("B7", "tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:205", B7_SRC, post=_post_B7)

### B6  `tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:105` (Track 08 ch09 — RMM managed memory; runnable after one documented replacement)

Run last because it re-initialises the RMM allocator for the whole process. **Documented replacement:** `initial_pool_size="64GB"` -> `"2GB"` (chapter value targets an 80 GB GPU). Verbatim run is not attempted on a 16 GB GPU.

In [ ]:
B6_SRC = r'''import rmm

# VRAM 80GB 초과 시 시스템 DDR5 RAM으로 자동 투명 페이징(Paging)
rmm.reinitialize(
    managed_memory=True,  # cudaMallocManaged 활성화
    pool_allocator=True,
    initial_pool_size="64GB"  # 문자열은 RMM이 직접 해석 (GB=10^9 bytes)
)
'''

run_block("B6", "tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:105", B6_SRC, replace=[('initial_pool_size="64GB"', 'initial_pool_size="2GB"')])

## 4. Write results

In [ ]:
import json, platform, subprocess
RESULTS["meta"] = {
    "date": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(),
    "gpu": GPU_NAME, "gpu_mem_gb": round(GPU_MEM_GB, 1), "versions": VERSIONS,
}
try:
    RESULTS["meta"]["nvidia_smi_head"] = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except Exception as e:
    RESULTS["meta"]["nvidia_smi_head"] = f"unavailable: {e}"
with open("gpu_results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False)
print(f"{'block':5s} {'status':10s} {'sec':>8s}  location")
for bid, r in RESULTS["blocks"].items():
    print(f"{bid:5s} {r['status']:10s} {r['elapsed_s']:>8}  {r['location']}" + (f"  -> {r['error'][:90]}" if r["error"] else ""))
n_pass = sum(r["status"] == "PASS" for r in RESULTS["blocks"].values())
print(f"\n{n_pass}/{len(RESULTS['blocks'])} blocks PASS. Results saved to gpu_results.json")
print(">>> Download gpu_results.json (Files pane on the left, or the download starting now) and send it back. <<<")
try:
    from google.colab import files
    files.download("gpu_results.json")
except Exception as e:
    print("(automatic download unavailable:", type(e).__name__, ") -> use the Files pane.")